# 04d — Text-Only Benchmarks

**Input:** `../data/raw/ema_personality_plus_surveys_merged.csv`
**Output:** `../outputs/tables/text_only_summary.csv`, `../outputs/tables/text_only_item_level.csv`

**Description:**
- Models using ONLY text embeddings (no numeric baseline) to establish what text alone can do
- Concurrent prediction: text → same-day AM crisis, text → same-day PM crisis
- Item-level: text → individual DSI-SS items A–D separately
- Ridge regression with PCA, GroupKFold CV, Pearson r of OOF predictions
- Matches original cells 11–12

In [7]:
import os
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sentence_transformers import SentenceTransformer

# =========================
# CONFIG
# =========================
DATA_PATH = os.path.join("..", "data", "raw", "ema_personality_plus_surveys_merged.csv")
OUT_DIR = os.path.join("..", "outputs", "tables")
os.makedirs(OUT_DIR, exist_ok=True)

PID_COL = "expiwell_id_clean"
DT_COL = "ema_dt"
TEXT_COL = "ema_text"

AM_TOTAL = "dailyAM__Total Score from 5 Questions"
PM_TOTAL = "dailyPM__Total Score from 5 Questions"

EMBED_MODEL = "sentence-transformers/all-mpnet-base-v2"
CACHE_DIR = os.path.join("..", "data", "processed")

N_SPLITS = 5
N_PCS = 20
RIDGE_ALPHA = 10.0
RANDOM_SEED = 7
np.random.seed(RANDOM_SEED)

In [8]:
# =========================
# LOAD + BUILD DAY TABLE
# =========================
df = pd.read_csv(DATA_PATH)

if PID_COL not in df.columns:
    PID_COL = "expiwell_id"
if DT_COL not in df.columns:
    DT_COL = "start_date"

df[DT_COL] = pd.to_datetime(df[DT_COL], errors="coerce")
df["ema_date"] = df[DT_COL].dt.date

df["crisis_AM"] = df[AM_TOTAL]
df["crisis_PM"] = df[PM_TOTAL]

def first_nonnull(x):
    x = x.dropna()
    return x.iloc[0] if len(x) else np.nan

day = (
    df.sort_values([PID_COL, DT_COL])
    .groupby([PID_COL, "ema_date"], as_index=False)
    .agg(
        day_text=(TEXT_COL, lambda s: " ".join([str(t) for t in s.dropna()])),
        n_text=(TEXT_COL, lambda s: int(s.notna().sum())),
        crisis_AM=("crisis_AM", first_nonnull),
        crisis_PM=("crisis_PM", first_nonnull),
    )
)

day["day_text"] = day["day_text"].fillna("").astype(str)
print("Day rows:", len(day), "| Participants:", day[PID_COL].nunique())

st_model = SentenceTransformer(EMBED_MODEL)

Day rows: 2734 | Participants: 123


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
# =========================
# TEXT-ONLY CV FUNCTION
# =========================
def text_only_cv(day_sub, text_col, y_col, label, embed_tag):
    day_sub = day_sub.copy()
    day_sub = day_sub.loc[
        day_sub[text_col].str.strip().ne("") & day_sub[y_col].notna()
    ].reset_index(drop=True)

    if len(day_sub) < 50:
        print(f"  {label}: too few rows ({len(day_sub)})")
        return None

    texts = day_sub[text_col].tolist()
    y = day_sub[y_col].values.astype(float)
    groups = day_sub[PID_COL].astype(str).values

    # Embeddings
    cache_path = os.path.join(CACHE_DIR, f"emb_textonly_{embed_tag}.npy")
    if os.path.exists(cache_path):
        X = np.load(cache_path)
        if X.shape[0] != len(texts):
            os.remove(cache_path)
            X = None
    else:
        X = None

    if X is None:
        X = st_model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
        np.save(cache_path, X)

    n_splits_eff = min(N_SPLITS, day_sub[PID_COL].nunique())
    gkf = GroupKFold(n_splits=n_splits_eff)
    yhat_oof = np.zeros_like(y, dtype=float)

    for tr, te in gkf.split(X, y, groups=groups):
        pca = PCA(n_components=min(N_PCS, X.shape[1]), random_state=RANDOM_SEED)
        Xtr = pca.fit_transform(X[tr])
        Xte = pca.transform(X[te])

        scaler = StandardScaler()
        Xtr = scaler.fit_transform(Xtr)
        Xte = scaler.transform(Xte)

        reg = Ridge(alpha=RIDGE_ALPHA, random_state=RANDOM_SEED)
        reg.fit(Xtr, y[tr])
        yhat_oof[te] = reg.predict(Xte)

    pearson_r = float(np.corrcoef(y, yhat_oof)[0, 1])
    mae = float(mean_absolute_error(y, yhat_oof))
    rmse = float(np.sqrt(mean_squared_error(y, yhat_oof)))
    r2 = float(r2_score(y, yhat_oof))

    print(f"  {label}: n={len(day_sub)} | r={pearson_r:.3f} | R²={r2:.3f} | MAE={mae:.3f} | RMSE={rmse:.3f}")

    return {
        "label": label, "n": len(day_sub),
        "participants": int(day_sub[PID_COL].nunique()),
        "pearson_r": pearson_r, "r2": r2, "mae": mae, "rmse": rmse,
    }

In [10]:
# =========================
# RUN TEXT-ONLY CONCURRENT MODELS
# =========================
results = []

print("=== Text-Only Concurrent Prediction ===")

r = text_only_cv(day, "day_text", "crisis_AM", "Text_to_AMcrisis_sameday", "am_concurrent")
if r:
    results.append(r)

r = text_only_cv(day, "day_text", "crisis_PM", "Text_to_PMcrisis_sameday", "pm_concurrent")
if r:
    results.append(r)

summary = pd.DataFrame(results)
summary_path = os.path.join(OUT_DIR, "text_only_summary.csv")
summary.to_csv(summary_path, index=False)
print("\nSaved:", summary_path)

=== Text-Only Concurrent Prediction ===
  Text_to_AMcrisis_sameday: n=2076 | r=0.323 | R²=0.096 | MAE=4.825 | RMSE=5.825


Batches:   0%|          | 0/43 [00:00<?, ?it/s]

  Text_to_PMcrisis_sameday: n=2730 | r=0.501 | R²=0.246 | MAE=4.325 | RMSE=5.384

Saved: ../outputs/tables/text_only_summary.csv


In [ ]:
# =========================
# ITEM-LEVEL: Text → individual DSI-SS items
# =========================
def find_dsi_cols_by_prefix(df, prefix):
    cols = [c for c in df.columns if c.startswith(prefix)]
    def pick(patterns):
        pats = [re.compile(p, re.IGNORECASE) for p in patterns]
        hits = [c for c in cols if any(p.search(c) for p in pats)]
        hits = sorted(hits, key=len)
        return hits[0] if hits else None
    A = pick([r"thoughts of killing myself", r"killing myself"])
    B = pick([r"definite plan", r"formulated.*plan", r"plans?"])
    C = pick([r"little or no control", r"control over", r"control"])
    D = pick([r"impulses to kill myself", r"impulses"])
    return {"A": A, "B": B, "C": C, "D": D}

DSI_PM = find_dsi_cols_by_prefix(df, "dailyPM__")
print("DSI PM items:", DSI_PM)

# Add DSI items to day table
for item_label, col_name in DSI_PM.items():
    if col_name is not None:
        day[f"dsi_{item_label}"] = df.groupby([PID_COL, "ema_date"])[col_name].first().reindex(
            pd.MultiIndex.from_frame(day[[PID_COL, "ema_date"]])
        ).values

print("\n=== Item-Level Prediction ===")
item_results = []
for item_label, col_name in DSI_PM.items():
    if col_name is None:
        continue
    r = text_only_cv(day, "day_text", f"dsi_{item_label}", f"Text_to_DSI_{item_label}", f"dsi_item_{item_label}")
    if r:
        item_results.append(r)

if item_results:
    item_df = pd.DataFrame(item_results)
    item_path = os.path.join(OUT_DIR, "text_only_item_level.csv")
    item_df.to_csv(item_path, index=False)
    print("\nSaved:", item_path)